# Mushroom Edibility — Data Cleaning & Feature Engineering

Pipeline:
1. Encode `class` (e -> 0, p -> 1)
2. Replace '?' in `stalk_root` with a new "missing" category (preserves the missingness signal)
3. Drop constant column(s) (`veil_type`)
4. One-hot encode all remaining categorical features (`drop_first=True`)
5. Save to `data/mushrooms_cleaned.csv`

## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append(".")
from utils import (load_data, encode_target, handle_missing,
                   drop_constant, preprocess_data)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

%matplotlib inline

In [2]:
df = load_data("data/mushrooms.csv")
print(f"Raw shape: {df.shape}")
df.head()

Raw shape: (8124, 23)


,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


## 2. Missing / Invalid Values

The UCI dataset uses `'?'` to mark missing `stalk_root` values.

In [3]:
missing_marker = (df == "?").sum()
print(missing_marker[missing_marker > 0])
print(f"\nDuplicate rows: {df.duplicated().sum()}")

stalk_root    2480
dtype: int64

Duplicate rows: 0


## 3. Encode Target

In [4]:
df_enc = encode_target(df)
print("Class distribution:")
print(df_enc["class"].value_counts())
df_enc.head()

Class distribution:
class
0    4208
1    3916
Name: count, dtype: int64


,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,...,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
0,1,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,0,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,0,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,1,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,0,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


## 4. Handle Missing Values

We treat missing `stalk_root` as its own category (`missing`) rather than imputing — the missingness itself may carry signal.

In [5]:
df_filled = handle_missing(df_enc, strategy="category")
print("After replacing '?' with 'missing':")
print(df_filled["stalk_root"].value_counts())

After replacing '?' with 'missing':
stalk_root
b          3776
missing    2480
e          1120
c           556
r           192
Name: count, dtype: int64


## 5. Drop Constant Columns

In [6]:
df_no_const, dropped = drop_constant(df_filled)
print(f"Dropped columns (constant): {dropped}")
print(f"Shape after dropping: {df_no_const.shape}")

Dropped columns (constant): ['veil_type']
Shape after dropping: (8124, 22)


## 6. Distribution Check Pre/Post Encoding

In [7]:
cat_cols = [c for c in df_no_const.columns if c != "class" and df_no_const[c].dtype == object]
print(f"Will one-hot encode {len(cat_cols)} categorical columns")
print(f"  -> total dummy columns:",
      sum(df_no_const[c].nunique() - 1 for c in cat_cols))

Will one-hot encode 21 categorical columns
  -> total dummy columns: 95


## 7. Run the Full Pipeline

In [8]:
df_processed = preprocess_data(df, missing_strategy="category")
print(f"Processed shape: {df_processed.shape}")
print(f"Missing values : {df_processed.isnull().sum().sum()}")
df_processed.head()

Processed shape: (8124, 96)
Missing values : 0


,class,cap_shape_c,cap_shape_f,cap_shape_k,cap_shape_s,cap_shape_x,cap_surface_g,cap_surface_s,cap_surface_y,cap_color_c,...,population_n,population_s,population_v,population_y,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
0,1,0,0,0,0,1,0,1,0,0,...,0,1,0,0,0,0,0,0,1,0
1,0,0,0,0,0,1,0,1,0,0,...,1,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,1,0,0,0
3,1,0,0,0,0,1,0,0,1,0,...,0,1,0,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


## 8. Sanity Checks & Save

In [9]:
assert df_processed["class"].isin([0, 1]).all()
assert df_processed.isnull().sum().sum() == 0
assert "veil_type" not in df_processed.columns
print("All checks passed.")

All checks passed.


In [10]:
df_processed.to_csv("data/mushrooms_cleaned.csv", index=False)
print(f"Saved data/mushrooms_cleaned.csv ({df_processed.shape[0]} rows, {df_processed.shape[1]} cols)")

Saved data/mushrooms_cleaned.csv (8124 rows, 96 cols)
